In [29]:
!wget -q https://raw.githubusercontent.com/cbuntain/umd.inst414/refs/heads/main/Module05/movie_to_cluster.csv
!wget https://raw.githubusercontent.com/cbuntain/umd.inst414/refs/heads/main/data/imdb_movies_2000to2022.prolific.json

--2026-10-30 11:49:08--  https://raw.githubusercontent.com/cbuntain/umd.inst414/refs/heads/main/data/imdb_movies_2000to2022.prolific.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6197619 (5.9M) [text/plain]
Saving to: ‘imdb_movies_2000to2022.prolific.json’

imdb_movies_2000to2 100%[===================>]   5.91M  --.-KB/s    in 0.1s    

2026-10-30 11:49:08 (51.0 MB/s) - ‘imdb_movies_2000to2022.prolific.json’ saved [6197619/6197619]



In [30]:
import json
import pandas as pd

actor_name_map = {} #{"actor_id": "actor_name"}
movie_actor_map = {} #{"ovie_id"}: "actor_id_list"}
with open("imdb_movies_2000to2022.prolific.json") as in_file:
    for line in in_file:
        this_movie = json.loads(line)
        for actor_id, actor_name in this_movie["actors"]:
            actor_name_map[actor_id] = actor_name
        movie_actor_map[this_movie["imdb_id"]] = set(items[0] for items in this_movie ["actors"])

In [31]:
cluster_df = pd.read_csv("movie_to_cluster.csv")
cluster_df.head()

,movie_id,cluster
0,tt0035423,8
1,tt0088751,12
2,tt0096056,6
3,tt0113092,3
4,tt0116391,3


In [32]:
movie_count_total = len(movie_actor_map)
actor_counts = {}
for actor_list in movie_actor_map.values():
    for actor in actor_list:
        actor_counts[actor] = actor_counts.get(actor, 0) + 1

In [33]:
#Pr(actor)
actor_pr_series = pd.Series(actor_counts) / movie_count_total
actor_pr_map = actor_pr_series.to_dict()
# actor_pr_map_oneLiner = {actor_id: count / movie_count_total for actor, count in actor_counts.items()}

for target_actor_id in ["nm0000115", "nm0000206", "nm1089991"]:
    print("Actor:", actor_name_map[target_actor_id])
    for cluster_id, cluster in cluster_df.groupby("cluster"):
        #Pr(Cluster)
        movie_count_in_the_cluster = len(cluster)
        pr_cluster = movie_count_in_the_cluster / movie_count_total
    
        #Pr(Actor, Cluster)
        this_cluster_actor_count = sum(target_actor_id in movie_actor_map[movie_id] for movie_id in cluster ["movie_id"])
        pr_actor_given_cluster = this_cluster_actor_count / movie_count_in_the_cluster
    
        #Pr(Actor, Cluster) = Pr(Actor | Cluster) * Pr(Cluster)
        joint_pr_actor_cluster = pr_actor_given_cluster * pr_cluster

        #Pr(Cluster | Actor) = Pr(Actor, Cluster) / Pr(Actor)
        pr_actor = actor_pr_map[target_actor_id]
        pr_cluster_given_actor = joint_pr_actor_cluster / pr_actor
        print("Pr[Cluster %02d | %s]:" % (cluster_id, target_actor_id), f"\t {round(pr_cluster_given_actor, 4)}")
    print()

Actor: Nicolas Cage
Pr[Cluster 00 | nm0000115]: 	 0.082
Pr[Cluster 01 | nm0000115]: 	 0.1311
Pr[Cluster 02 | nm0000115]: 	 0.1639
Pr[Cluster 03 | nm0000115]: 	 0.0984
Pr[Cluster 04 | nm0000115]: 	 0.0164
Pr[Cluster 05 | nm0000115]: 	 0.0
Pr[Cluster 06 | nm0000115]: 	 0.0328
Pr[Cluster 07 | nm0000115]: 	 0.0
Pr[Cluster 08 | nm0000115]: 	 0.0
Pr[Cluster 09 | nm0000115]: 	 0.0492
Pr[Cluster 10 | nm0000115]: 	 0.0
Pr[Cluster 11 | nm0000115]: 	 0.0328
Pr[Cluster 12 | nm0000115]: 	 0.0328
Pr[Cluster 13 | nm0000115]: 	 0.0656
Pr[Cluster 14 | nm0000115]: 	 0.0164
Pr[Cluster 15 | nm0000115]: 	 0.2787

Actor: Keanu Reeves
Pr[Cluster 00 | nm0000206]: 	 0.0938
Pr[Cluster 01 | nm0000206]: 	 0.0312
Pr[Cluster 02 | nm0000206]: 	 0.0938
Pr[Cluster 03 | nm0000206]: 	 0.0938
Pr[Cluster 04 | nm0000206]: 	 0.0938
Pr[Cluster 05 | nm0000206]: 	 0.0
Pr[Cluster 06 | nm0000206]: 	 0.0938
Pr[Cluster 07 | nm0000206]: 	 0.0625
Pr[Cluster 08 | nm0000206]: 	 0.0
Pr[Cluster 09 | nm0000206]: 	 0.0312
Pr[Cluster 10 | 